# Project AEON: From-Scratch JEPA Crafter World Model & Agent (Colab GPU Runner)

This notebook trains and validates the **AEON JEPA** world model on raw Crafter visual interaction (`64×64×3` RGB).

### Non-Negotiable Integrity Invariants:
1. **Zero Pretrained Weights:** 100% learned parameters initialized randomly.
2. **Zero Game Knowledge:** No crafting recipes, no inventory requirements, no object identities, no semantic action descriptions.
3. **Anonymous Action Space:** The agent sees only anonymous tokens `0..16`, translated via an opaque permutation.
4. **Information Quarantine:** Crafter `info` dictionaries are completely stripped from the learner and routed strictly to metrics evaluation.
5. **Persistent Drive Storage:** Experience, replay buffer, and model checkpoints stream directly to `/content/drive/MyDrive/learn-a-world/aeon/` to survive Colab disconnections.

## 1. Mount Google Drive and Setup Dedicated AEON Subfolder

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure dedicated AEON model directory under learn-a-world
MASTER_DRIVE_DIR = Path('/content/drive/MyDrive/learn-a-world')
AEON_DRIVE_DIR = MASTER_DRIVE_DIR / 'aeon'

# Subdirectories for checkpoints, run logs, artifacts, and evaluation reports
for sub in ['checkpoints', 'runs', 'artifacts', 'results']:
    (AEON_DRIVE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print(f"Master Drive root: {MASTER_DRIVE_DIR}")
print(f"AEON dedicated folder: {AEON_DRIVE_DIR}")

## 2. Sync Repository from GitHub

In [ ]:
%cd /content
import os

# Clone or pull latest main branch
if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
else:
    %cd /content/learn-a-world
    !git fetch origin
    !git pull origin main

%cd /content/learn-a-world
!git log -1 --oneline

## 3. Install Dependencies & Hardware Audit

In [ ]:
# Install lightweight requirements
!pip install --quiet -r AEON/jepa-crafter/requirements.txt

# Set PYTHONPATH to include AEON/jepa-crafter
import sys
aeon_src_dir = '/content/learn-a-world/AEON/jepa-crafter'
if aeon_src_dir not in sys.path:
    sys.path.insert(0, aeon_src_dir)

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 4. Run Architectural & Information Integrity Test Suite

Runs all 10 acceptance tests verifying:
- 64x64 uint8 RGB observation contract
- Zero information leakage from Crafter internals
- Exact sequence alignment across horizons 1, 2, 4, 8
- Stop-gradient EMA target encoder behavior
- Cross-Entropy Method latent replanning

In [ ]:
%cd /content/learn-a-world/AEON/jepa-crafter
!python3 -m pytest -v tests/

## 5. Launch Training Campaign with Direct Google Drive Persistence

Trains the full AEON agent system:
- Initial random warm-up
- Multi-horizon JEPA future latent prediction with EMA target network
- Automated validation gates (Test B heldout prediction, Test C action-conditioning, Test D collapse checks)
- Staged curiosity exploration and Cross-Entropy Method (CEM) latent model-predictive control (MPC)
- All checkpoints and logs save atomically to `/content/drive/MyDrive/learn-a-world/aeon/`

In [ ]:
# Execute training run saving directly to Google Drive (Phase 4: 50,000 steps pilot)
%cd /content/learn-a-world
!python3 AEON/jepa-crafter/run.py \n  --config AEON/jepa-crafter/configs/crafter_t4.yaml \n  --drive-root /content/drive/MyDrive/learn-a-world/aeon \n  --max-steps 50000 \n  --num-envs 8 \n  --device auto


## 6. Inspect Results and Benchmark Scorecard

Loads the generated `final_report.md` and displays the official Crafter 22-achievement scorecard.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

aeon_run_dir = Path("/content/drive/MyDrive/learn-a-world/aeon/runs/jepa_crafter")

# 1. Display Final Research Report
report_path = aeon_run_dir / "final_report.md"
if report_path.exists():
    display(Markdown(report_path.read_text()))
else:
    print(f"Report not yet generated at {report_path}")

# 2. Display Achievement Rates Table
csv_path = aeon_run_dir / "achievement_rates.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("\n=== OFFICIAL CRAFTER BENCHMARK SCORECARD ===")
    display(df)
